# ================================================================
# STEP 8 — DASHBOARD & THREAT ANALYTICS PIPELINE
# ================================================================

## PROJECT PHASE TRANSITION

Step 1–7: COMPLETED & ARCHIVED

The Unsupervised ML pipeline has been completed and validated.

Primary production model:
- DBSCAN
- eps = 0.5
- min_samples = 20
- Non-noise behavioral clusters = 38
- Noise sessions = 1,676
- Behavioral features = 48
- Stability validation = PASS
- Production assignment pipeline = PASS

Step 7 Final Archive:
- Archive: /content/STEP_7_FINAL_ARCHIVE.zip
- Archive SHA-256:
  38767175fb702691eb6dd124f7017946500b3ff890e1da2cceddfdeaf5c8651a

Source Dataset SHA-256:
b9a5cdfd2dcc3369e5831ed4c3306f548fe48d951f216f15e71967de1e418534

----------------------------------------------------------------

## STEP 8 OBJECTIVE

Build the analytical and dashboard-ready layer on top of the
validated Unified Events and production behavioral clustering
pipeline.

This phase focuses on:

1. Source Freeze & Validation
2. GeoIP Analytics / Enrichment
3. Attacker IP Analytics
4. MITRE ATT&CK Analytics
5. Session Replay
6. Behavioral Cluster Analytics
7. Attack Trends & Statistics
8. Dashboard-ready datasets
9. Final Dashboard Validation

----------------------------------------------------------------

## IMPORTANT DATA GOVERNANCE RULES

- Never modify the frozen source dataset.
- Never overwrite Step 7 production artifacts.
- Never use attack_type or other labels as ML behavioral features.
- Never introduce cluster-derived features back into clustering.
- Preserve reconstructed_session_id integrity.
- Preserve the validated DBSCAN production reference.
- All new derived datasets must have their own validation and SHA-256.
- Existing Step 7 artifacts are treated as immutable production references.

----------------------------------------------------------------

## STEP 7 STATUS

STEP 7 = COMPLETE
STATUS = PASS

Step 8 begins here.

----------------------------------------------------------------

## STEP 8 PIPELINE

8A — Source Freeze + Schema Validation
8B — GeoIP Analytics / Enrichment
8C — Attacker IP Analytics
8D — MITRE ATT&CK Analytics
8E — Session Replay Dataset
8F — Behavioral Cluster Dashboard Data
8G — Attack Trends & Threat Statistics
8H — Dashboard Integration Dataset
8I — Final Dashboard Validation & Archive

----------------------------------------------------------------

## PRINCIPLE

Unified Events
      ↓
Validated Analytics Layer
      ↓
GeoIP + MITRE + Session Replay
      ↓
Behavioral Clusters
      ↓
Threat Analytics
      ↓
Dashboard-ready Data
      ↓
Final Dashboard

================================================================
STEP 8 — START
================================================================

In [ ]:
# ======================================================================
# STEP 8A-CLEAN — UNIFIED EVENTS SOURCE FREEZE + IDENTITY AUDIT
# ======================================================================

from pathlib import Path
import pandas as pd
import hashlib
import json
import shutil

print("=" * 72)
print("STEP 8A-CLEAN — UNIFIED EVENTS SOURCE FREEZE + IDENTITY AUDIT")
print("=" * 72)

# ----------------------------------------------------------------------
# 1. CONFIG
# ----------------------------------------------------------------------

SOURCE_NAME = "unified_security_events_clean_v1_BACKUP.csv"

SOURCE_PATH = Path("/content") / SOURCE_NAME

OUTPUT_DIR = Path(
    "/content/step8_dashboard_geoip_mitre_session_replay/source_freeze"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FROZEN_PATH = (
    OUTPUT_DIR /
    "unified_security_events_step8_frozen_v1.csv"
)

MANIFEST_PATH = (
    OUTPUT_DIR /
    "step8a_unified_events_source_manifest.json"
)


# ----------------------------------------------------------------------
# 2. EXPECTED IDENTITY
# ----------------------------------------------------------------------
# IMPORTANT:
# This is NOT the Step 7 SHA.
# It is the SHA of the Unified Events backup currently being used
# as the independent Step 8 source.

EXPECTED_SHA = (
    "a1639f172eaca4a3c3a4ed02045b33e7d8152c5ec71129e7d33ad8f479edbcca"
)


# ----------------------------------------------------------------------
# 3. HELPERS
# ----------------------------------------------------------------------

def sha256_file(path, chunk_size=1024 * 1024):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        while True:

            chunk = f.read(chunk_size)

            if not chunk:
                break

            h.update(chunk)

    return h.hexdigest()


def stop(message):

    raise ValueError(
        "\n\nSTOP — " + message
    )


# ----------------------------------------------------------------------
# 4. SOURCE EXISTENCE
# ----------------------------------------------------------------------

print("\n[1] SOURCE FILE")

if not SOURCE_PATH.exists():

    stop(
        f"Unified Events source not found:\n{SOURCE_PATH}"
    )

print(f"Path: {SOURCE_PATH}")
print("File exists: PASS")


# ----------------------------------------------------------------------
# 5. SOURCE HASH
# ----------------------------------------------------------------------

print("\n[2] SOURCE IDENTITY")

actual_sha = sha256_file(SOURCE_PATH)

print(f"Expected SHA: {EXPECTED_SHA}")
print(f"Actual SHA:   {actual_sha}")

if actual_sha != EXPECTED_SHA:

    stop(
        "Unified Events source SHA mismatch.\n"
        "This file is not the verified Step 8 source."
    )

print("SHA verification: PASS")


# ----------------------------------------------------------------------
# 6. LOAD DATASET
# ----------------------------------------------------------------------

print("\n[3] LOADING UNIFIED EVENTS")

df = pd.read_csv(SOURCE_PATH)

print(f"Shape: {df.shape}")
print(f"Rows: {len(df)}")
print(f"Columns: {len(df.columns)}")


# ----------------------------------------------------------------------
# 7. BASIC DATASET SANITY
# ----------------------------------------------------------------------

print("\n[4] DATASET SANITY")

if len(df) == 0:

    stop("Unified Events dataset is empty.")

if len(df.columns) == 0:

    stop("Unified Events dataset has no columns.")

print("Dataset non-empty: PASS")


# ----------------------------------------------------------------------
# 8. SCHEMA INVENTORY
# ----------------------------------------------------------------------

print("\n[5] SCHEMA INVENTORY")

print("Columns:")

for i, col in enumerate(df.columns, start=1):

    print(f"{i:02d}. {col}")


# ----------------------------------------------------------------------
# 9. ID COLUMN DETECTION
# ----------------------------------------------------------------------

print("\n[6] ID / EVENT IDENTITY")

id_candidates = [
    "event_id",
    "reconstructed_session_id",
    "session_id",
    "id"
]

detected_ids = [
    col for col in id_candidates
    if col in df.columns
]

print(f"Candidate ID columns found: {detected_ids}")


# We do NOT force a reconstructed session ID here.
# Step 8 will reconstruct sessions independently.

session_id_column = None

for col in [
    "reconstructed_session_id",
    "session_id"
]:

    if col in df.columns:

        session_id_column = col
        break


if session_id_column:

    print(f"Existing session column: {session_id_column}")

    missing_session_ids = int(
        df[session_id_column].isna().sum()
    )

    duplicate_session_ids = int(
        df[session_id_column].duplicated().sum()
    )

    print(
        f"Missing session IDs: {missing_session_ids}"
    )

    print(
        f"Duplicate session IDs: {duplicate_session_ids}"
    )

else:

    print(
        "No reconstructed session ID found."
    )

    print(
        "This is ACCEPTED — Step 8 will rebuild sessions."
    )


# ----------------------------------------------------------------------
# 10. EVENT ID VALIDATION
# ----------------------------------------------------------------------

event_id_column = None

for col in ["event_id", "id"]:

    if col in df.columns:

        event_id_column = col
        break


if event_id_column:

    missing_event_ids = int(
        df[event_id_column].isna().sum()
    )

    duplicate_event_ids = int(
        df[event_id_column].duplicated().sum()
    )

    unique_event_ids = int(
        df[event_id_column].nunique()
    )

    print(f"\nEvent ID column: {event_id_column}")
    print(f"Missing event IDs: {missing_event_ids}")
    print(f"Duplicate event IDs: {duplicate_event_ids}")
    print(f"Unique event IDs: {unique_event_ids}")

    if missing_event_ids > 0:

        stop(
            "Event ID column contains missing values."
        )

else:

    print(
        "\nNo explicit event_id column found."
    )

    print(
        "Accepted for now; Step 8 will audit event identity "
        "using the available Unified Events schema."
    )


# ----------------------------------------------------------------------
# 11. DUPLICATE ROW AUDIT
# ----------------------------------------------------------------------

print("\n[7] DUPLICATE ROW AUDIT")

duplicate_rows = int(
    df.duplicated().sum()
)

print(f"Duplicate complete rows: {duplicate_rows}")

if duplicate_rows > 0:

    print(
        "WARNING — Duplicate complete rows detected."
    )

    print(
        "They will NOT be removed in Step 8A."
    )

    print(
        "Deduplication will be handled explicitly in the "
        "appropriate downstream validation stage."
    )

else:

    print("Duplicate complete rows: 0")


# ----------------------------------------------------------------------
# 12. MISSINGNESS PROFILE
# ----------------------------------------------------------------------

print("\n[8] MISSINGNESS PROFILE")

missing_counts = df.isna().sum()

missing_columns = (
    missing_counts[missing_counts > 0]
    .sort_values(ascending=False)
)

print(
    f"Columns containing missing values: "
    f"{len(missing_columns)}"
)

if len(missing_columns) > 0:

    print("\nTop missingness columns:")

    print(
        missing_columns.head(20).to_string()
    )

else:

    print("No missing values detected.")


# ----------------------------------------------------------------------
# 13. FREEZE SOURCE
# ----------------------------------------------------------------------

print("\n[9] CREATING STEP 8 FROZEN SOURCE")

shutil.copy2(
    SOURCE_PATH,
    FROZEN_PATH
)

frozen_sha = sha256_file(FROZEN_PATH)

print(f"Frozen path: {FROZEN_PATH}")
print(f"Frozen SHA:  {frozen_sha}")

if frozen_sha != EXPECTED_SHA:

    stop(
        "Frozen Step 8 source SHA does not match "
        "the verified Unified Events SHA."
    )

print("Frozen source: PASS")


# ----------------------------------------------------------------------
# 14. BUILD MANIFEST
# ----------------------------------------------------------------------

print("\n[10] BUILDING SOURCE MANIFEST")

manifest = {

    "step": "8A-CLEAN",

    "status": "PASS",

    "source_role":
        "Canonical Unified Events input for Step 8",

    "source_path":
        str(SOURCE_PATH),

    "frozen_source_path":
        str(FROZEN_PATH),

    "source_sha256":
        actual_sha,

    "frozen_sha256":
        frozen_sha,

    "row_count":
        int(len(df)),

    "column_count":
        int(len(df.columns)),

    "columns":
        list(df.columns),

    "event_id_column":
        event_id_column,

    "existing_session_id_column":
        session_id_column,

    "duplicate_complete_rows":
        duplicate_rows,

    "missingness_columns":
        {
            str(k): int(v)
            for k, v in missing_counts.items()
            if v > 0
        },

    "step7_source_sha256":
        "b9a5cdfd2dcc3369e5831ed4c3306f548fe48d951f216f15e71967de1e418534",

    "step7_sha_used_for_identity":
        False,

    "important_note":
        "Step 8 is an independent analytical pipeline "
        "starting from the Unified Events source. "
        "Step 7 artifacts remain frozen separately.",

    "next_step":
        "STEP 8B — GEOIP ENRICHMENT"
}


with open(
    MANIFEST_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        manifest,
        f,
        indent=2,
        ensure_ascii=False
    )


print(f"Manifest: {MANIFEST_PATH}")


# ----------------------------------------------------------------------
# 15. FINAL PROTECTION CHECK
# ----------------------------------------------------------------------

print("\n[11] FINAL PROTECTION CHECK")

source_sha_after = sha256_file(SOURCE_PATH)
frozen_sha_after = sha256_file(FROZEN_PATH)

print(
    f"Original source SHA: {source_sha_after}"
)

print(
    f"Frozen source SHA:   {frozen_sha_after}"
)

if source_sha_after != EXPECTED_SHA:

    stop(
        "Original Unified Events source changed "
        "during Step 8A."
    )

if frozen_sha_after != EXPECTED_SHA:

    stop(
        "Frozen Unified Events source changed."
    )

print("Source unchanged: PASS")
print("Frozen copy integrity: PASS")


# ----------------------------------------------------------------------
# FINAL
# ----------------------------------------------------------------------

print("\n" + "=" * 72)
print("STEP 8A-CLEAN — COMPLETE")
print("=" * 72)

print(f"Source rows: {len(df)}")
print(f"Source columns: {len(df.columns)}")
print(f"Source SHA-256: {actual_sha}")

print("\nFrozen source:")
print(FROZEN_PATH)

print("\nManifest:")
print(MANIFEST_PATH)

print("\nSTATUS: PASS")
print("NEXT: STEP 8B — GEOIP ENRICHMENT")
print("=" * 72)

In [ ]:
# ================================================================
# STEP 8B — PRODUCTION GEOIP ENRICHMENT + IP LOOKUP DATABASE
# ================================================================
#
# Purpose:
#   1. Load the frozen Step 8 source.
#   2. Locate GeoLite2-City and GeoLite2-ASN databases.
#   3. Build a persistent SQLite IP GeoIP cache.
#   4. Enrich source_ip for all current events.
#   5. Make the cache reusable for future/new IPs.
#   6. Validate coverage and consistency.
#   7. Freeze outputs + SHA-256 + manifest + ZIP.
#
# IMPORTANT:
#   - Frozen source is NEVER modified.
#   - Step 7 artifacts are NEVER modified.
#   - GeoIP fields are enrichment only.
# ================================================================

from pathlib import Path
import hashlib
import json
import shutil
import sqlite3
import tarfile
import zipfile
import subprocess
import sys
import os
import re
from datetime import datetime, timezone

import numpy as np
import pandas as pd


# ================================================================
# 0. CONFIGURATION
# ================================================================

BASE = Path("/content")

STEP8_ROOT = BASE / "step8_dashboard_geoip_mitre_session_replay"
SOURCE_PATH = (
    STEP8_ROOT
    / "source_freeze"
    / "unified_security_events_step8_frozen_v1.csv"
)

OUT_DIR = STEP8_ROOT / "8B_geoip_enrichment"
DB_DIR = OUT_DIR / "geoip_database"
ARCHIVE_DIR = OUT_DIR / "archive"

OUT_DIR.mkdir(parents=True, exist_ok=True)
DB_DIR.mkdir(parents=True, exist_ok=True)
ARCHIVE_DIR.mkdir(parents=True, exist_ok=True)


CITY_ARCHIVE_PATTERN = "GeoLite2-City"
ASN_ARCHIVE_PATTERN = "GeoLite2-ASN"

GEOIP_SQLITE = DB_DIR / "ip_geoip_cache.sqlite"

ENRICHED_CSV = OUT_DIR / "geo_enriched_v2.csv"
IP_REFERENCE_CSV = OUT_DIR / "ip_geoip_reference.csv"
COVERAGE_CSV = OUT_DIR / "geoip_coverage_report.csv"
AUDIT_JSON = OUT_DIR / "step8b_geoip_enrichment_audit.json"
MANIFEST_JSON = OUT_DIR / "step8b_geoip_manifest.json"

FINAL_ZIP = ARCHIVE_DIR / "STEP_8B_GEOIP_ENRICHMENT.zip"


# ================================================================
# 1. HELPERS
# ================================================================

def sha256_file(path, chunk_size=1024 * 1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()


def require_file(path, label):
    if not path.exists():
        raise FileNotFoundError(f"STOP — {label} missing:\n{path}")


def print_header(title):
    print("\n" + "=" * 70)
    print(title)
    print("=" * 70)


def safe_str(value):
    if value is None:
        return None
    try:
        if pd.isna(value):
            return None
    except Exception:
        pass
    return str(value)


# ================================================================
# 2. SOURCE VERIFICATION
# ================================================================

print_header("STEP 8B — PRODUCTION GEOIP ENRICHMENT")

require_file(SOURCE_PATH, "Frozen Step 8 source")

source_sha = sha256_file(SOURCE_PATH)

print("[1] SOURCE VERIFICATION")
print("Frozen source:")
print(SOURCE_PATH)
print("SHA-256:", source_sha)

# Verify against 8A manifest if available.
SOURCE_MANIFEST = (
    STEP8_ROOT
    / "source_freeze"
    / "step8a_unified_events_source_manifest.json"
)

if SOURCE_MANIFEST.exists():
    with open(SOURCE_MANIFEST, "r", encoding="utf-8") as f:
        source_manifest = json.load(f)

    expected_sha = (
        source_manifest.get("source_sha256")
        or source_manifest.get("frozen_sha256")
        or source_manifest.get("sha256")
    )

    if expected_sha:
        if source_sha != expected_sha:
            raise ValueError(
                "STOP — Frozen source SHA does not match Step 8A manifest."
            )

        print("Manifest SHA:", expected_sha)
        print("SHA verification: PASS")
    else:
        print("WARNING — Manifest found but SHA field not detected.")
else:
    print("WARNING — Step 8A manifest not found; current frozen SHA recorded.")


# ================================================================
# 3. LOAD SOURCE
# ================================================================

print("\n[2] LOADING FROZEN SOURCE")

df = pd.read_csv(
    SOURCE_PATH,
    low_memory=False
)

print("Shape:", df.shape)
print("Rows:", len(df))
print("Columns:", len(df.columns))

if "source_ip" not in df.columns:
    raise ValueError("STOP — source_ip column is required.")

if df.empty:
    raise ValueError("STOP — frozen source is empty.")


# ================================================================
# 4. LOCATE GEOIP ARCHIVES
# ================================================================

print("\n[3] SEARCHING FOR GEOLITE2 DATABASES")

all_archives = list(BASE.rglob("*.tar.gz"))

city_archives = [
    p for p in all_archives
    if CITY_ARCHIVE_PATTERN.lower() in p.name.lower()
]

asn_archives = [
    p for p in all_archives
    if ASN_ARCHIVE_PATTERN.lower() in p.name.lower()
]

if not city_archives:
    raise FileNotFoundError(
        "STOP — GeoLite2-City .tar.gz was not found under /content."
    )

if not asn_archives:
    raise FileNotFoundError(
        "STOP — GeoLite2-ASN .tar.gz was not found under /content."
    )

# Prefer newest archive by modification time.
city_archive = sorted(
    city_archives,
    key=lambda p: p.stat().st_mtime,
    reverse=True
)[0]

asn_archive = sorted(
    asn_archives,
    key=lambda p: p.stat().st_mtime,
    reverse=True
)[0]

print("City archive:")
print(city_archive)
print("ASN archive:")
print(asn_archive)

city_archive_sha = sha256_file(city_archive)
asn_archive_sha = sha256_file(asn_archive)

print("City archive SHA-256:", city_archive_sha)
print("ASN archive SHA-256:", asn_archive_sha)


# ================================================================
# 5. EXTRACT MMDB FILES
# ================================================================

print("\n[4] EXTRACTING MAXMIND DATABASES")

CITY_EXTRACT = DB_DIR / "city_extract"
ASN_EXTRACT = DB_DIR / "asn_extract"

CITY_EXTRACT.mkdir(exist_ok=True)
ASN_EXTRACT.mkdir(exist_ok=True)


def extract_archive(archive, destination):
    with tarfile.open(archive, "r:gz") as tar:
        tar.extractall(destination)


extract_archive(city_archive, CITY_EXTRACT)
extract_archive(asn_archive, ASN_EXTRACT)


city_mmdb_candidates = list(CITY_EXTRACT.rglob("GeoLite2-City.mmdb"))
asn_mmdb_candidates = list(ASN_EXTRACT.rglob("GeoLite2-ASN.mmdb"))

if not city_mmdb_candidates:
    raise FileNotFoundError("STOP — GeoLite2-City.mmdb not found.")

if not asn_mmdb_candidates:
    raise FileNotFoundError("STOP — GeoLite2-ASN.mmdb not found.")

CITY_MMDB = city_mmdb_candidates[0]
ASN_MMDB = asn_mmdb_candidates[0]

print("City MMDB:", CITY_MMDB)
print("ASN MMDB:", ASN_MMDB)


# ================================================================
# 6. INSTALL / LOAD GEOIP2
# ================================================================

print("\n[5] GEOIP2 LIBRARY")

try:
    import geoip2.database
except ImportError:
    print("geoip2 not installed. Installing...")
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "geoip2"]
    )
    import geoip2.database

print("geoip2: READY")


# ================================================================
# 7. BUILD / OPEN PERSISTENT SQLITE GEOIP CACHE
# ================================================================

print("\n[6] BUILDING PERSISTENT IP GEOIP DATABASE")

conn = sqlite3.connect(GEOIP_SQLITE)

conn.execute("""
CREATE TABLE IF NOT EXISTS ip_geoip_cache (
    source_ip TEXT PRIMARY KEY,
    country TEXT,
    country_code TEXT,
    city TEXT,
    latitude REAL,
    longitude REAL,
    asn INTEGER,
    asn_org TEXT,
    geoip_status TEXT,
    city_lookup_status TEXT,
    asn_lookup_status TEXT,
    first_seen_utc TEXT,
    last_seen_utc TEXT,
    lookup_timestamp_utc TEXT
)
""")

conn.execute("""
CREATE INDEX IF NOT EXISTS idx_country_code
ON ip_geoip_cache(country_code)
""")

conn.execute("""
CREATE INDEX IF NOT EXISTS idx_asn
ON ip_geoip_cache(asn)
""")

conn.commit()

print("SQLite cache:", GEOIP_SQLITE)


# ================================================================
# 8. EXTRACT UNIQUE SOURCE IPS
# ================================================================

print("\n[7] SOURCE IP INVENTORY")

source_ips = (
    df["source_ip"]
    .dropna()
    .astype(str)
    .str.strip()
)

source_ips = source_ips[
    (source_ips != "") &
    (source_ips.str.lower() != "nan")
]

unique_ips = sorted(source_ips.unique())

print("Events:", len(df))
print("Events with source_ip:", len(source_ips))
print("Unique source IPs:", len(unique_ips))


# ================================================================
# 9. GEOIP LOOKUP ENGINE
# ================================================================

print("\n[8] GEOIP LOOKUP")

city_reader = geoip2.database.Reader(str(CITY_MMDB))
asn_reader = geoip2.database.Reader(str(ASN_MMDB))

lookup_timestamp = datetime.now(timezone.utc).isoformat()

lookup_rows = []

city_success = 0
asn_success = 0
city_fail = 0
asn_fail = 0
cache_hits = 0
new_ip_lookups = 0


def cached_ip(ip):
    row = conn.execute(
        """
        SELECT
            source_ip,
            country,
            country_code,
            city,
            latitude,
            longitude,
            asn,
            asn_org,
            geoip_status,
            city_lookup_status,
            asn_lookup_status,
            first_seen_utc,
            last_seen_utc,
            lookup_timestamp_utc
        FROM ip_geoip_cache
        WHERE source_ip = ?
        """,
        (ip,)
    ).fetchone()

    return row


def lookup_new_ip(ip):

    global city_success, city_fail
    global asn_success, asn_fail

    country = None
    country_code = None
    city = None
    latitude = None
    longitude = None
    asn = None
    asn_org = None

    city_status = "NOT_LOOKED_UP"
    asn_status = "NOT_LOOKED_UP"

    # ---------------------------
    # CITY
    # ---------------------------
    try:
        result = city_reader.city(ip)

        country = result.country.name
        country_code = result.country.iso_code
        city = result.city.name
        latitude = result.location.latitude
        longitude = result.location.longitude

        city_status = "FOUND"
        city_success += 1

    except Exception as e:
        city_status = type(e).__name__
        city_fail += 1

    # ---------------------------
    # ASN
    # ---------------------------
    try:
        result = asn_reader.asn(ip)

        asn = result.autonomous_system_number
        asn_org = result.autonomous_system_organization

        asn_status = "FOUND"
        asn_success += 1

    except Exception as e:
        asn_status = type(e).__name__
        asn_fail += 1

    now = datetime.now(timezone.utc).isoformat()

    return {
        "source_ip": ip,
        "country": country,
        "country_code": country_code,
        "city": city,
        "latitude": latitude,
        "longitude": longitude,
        "asn": asn,
        "asn_org": asn_org,
        "geoip_status": (
            "FOUND"
            if city_status == "FOUND" or asn_status == "FOUND"
            else "NOT_FOUND"
        ),
        "city_lookup_status": city_status,
        "asn_lookup_status": asn_status,
        "first_seen_utc": now,
        "last_seen_utc": now,
        "lookup_timestamp_utc": lookup_timestamp,
    }


for i, ip in enumerate(unique_ips, start=1):

    cached = cached_ip(ip)

    if cached is not None:

        cache_hits += 1

        lookup_rows.append({
            "source_ip": cached[0],
            "country": cached[1],
            "country_code": cached[2],
            "city": cached[3],
            "latitude": cached[4],
            "longitude": cached[5],
            "asn": cached[6],
            "asn_org": cached[7],
            "geoip_status": cached[8],
            "city_lookup_status": cached[9],
            "asn_lookup_status": cached[10],
            "first_seen_utc": cached[11],
            "last_seen_utc": lookup_timestamp,
            "lookup_timestamp_utc": cached[13],
        })

        conn.execute(
            """
            UPDATE ip_geoip_cache
            SET last_seen_utc = ?
            WHERE source_ip = ?
            """,
            (lookup_timestamp, ip)
        )

    else:

        new_ip_lookups += 1

        row = lookup_new_ip(ip)
        lookup_rows.append(row)

        conn.execute(
            """
            INSERT OR REPLACE INTO ip_geoip_cache (
                source_ip,
                country,
                country_code,
                city,
                latitude,
                longitude,
                asn,
                asn_org,
                geoip_status,
                city_lookup_status,
                asn_lookup_status,
                first_seen_utc,
                last_seen_utc,
                lookup_timestamp_utc
            )
            VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
            """,
            (
                row["source_ip"],
                row["country"],
                row["country_code"],
                row["city"],
                row["latitude"],
                row["longitude"],
                row["asn"],
                row["asn_org"],
                row["geoip_status"],
                row["city_lookup_status"],
                row["asn_lookup_status"],
                row["first_seen_utc"],
                row["last_seen_utc"],
                row["lookup_timestamp_utc"],
            )
        )

    if i % 500 == 0:
        conn.commit()
        print(f"Processed {i:,}/{len(unique_ips):,} IPs")

conn.commit()

city_reader.close()
asn_reader.close()

print("Lookup complete.")
print("Cache hits:", cache_hits)
print("New IP lookups:", new_ip_lookups)
print("City success:", city_success)
print("City failures:", city_fail)
print("ASN success:", asn_success)
print("ASN failures:", asn_fail)


# ================================================================
# 10. BUILD IP REFERENCE DATASET
# ================================================================

print("\n[9] BUILDING IP GEOIP REFERENCE")

ip_ref = pd.DataFrame(lookup_rows)

ip_ref = ip_ref.drop_duplicates(
    subset=["source_ip"],
    keep="last"
)

ip_ref.to_csv(
    IP_REFERENCE_CSV,
    index=False
)

print("Saved:", IP_REFERENCE_CSV)
print("Unique IP records:", len(ip_ref))


# ================================================================
# 11. ENRICH EVENTS
# ================================================================

print("\n[10] ENRICHING EVENTS")

geo_columns = [
    "source_ip",
    "country",
    "country_code",
    "city",
    "latitude",
    "longitude",
    "asn",
    "asn_org",
    "geoip_status",
]

geo_for_merge = ip_ref[geo_columns].copy()

# Preserve original source fields.
# New GeoIP fields use explicit names where necessary.
geo_for_merge = geo_for_merge.rename(columns={
    "country": "geoip_country",
    "country_code": "geoip_country_code",
    "city": "geoip_city",
    "latitude": "geoip_latitude",
    "longitude": "geoip_longitude",
    "asn": "geoip_asn",
    "asn_org": "geoip_asn_org",
    "geoip_status": "geoip_status",
})

enriched = df.merge(
    geo_for_merge,
    on="source_ip",
    how="left",
    validate="many_to_one"
)

print("Enriched shape:", enriched.shape)


# ================================================================
# 12. COVERAGE AUDIT
# ================================================================

print("\n[11] GEOIP COVERAGE AUDIT")

total_events = len(enriched)

coverage_rows = []

for field in [
    "geoip_country",
    "geoip_country_code",
    "geoip_city",
    "geoip_latitude",
    "geoip_longitude",
    "geoip_asn",
    "geoip_asn_org",
]:

    non_null = enriched[field].notna().sum()
    coverage = non_null / total_events if total_events else 0

    coverage_rows.append({
        "field": field,
        "non_null_events": int(non_null),
        "missing_events": int(total_events - non_null),
        "coverage_fraction": coverage,
        "coverage_percent": coverage * 100,
    })

coverage_df = pd.DataFrame(coverage_rows)

coverage_df.to_csv(
    COVERAGE_CSV,
    index=False
)

print(coverage_df.to_string(index=False))


# ================================================================
# 13. CONSISTENCY CHECKS
# ================================================================

print("\n[12] CONSISTENCY VALIDATION")

# source_ip must remain unchanged.
original_ips = df["source_ip"].astype("string")
enriched_ips = enriched["source_ip"].astype("string")

if not original_ips.equals(enriched_ips):
    raise ValueError(
        "STOP — source_ip order/content changed during enrichment."
    )

# Row count must remain identical.
if len(enriched) != len(df):
    raise ValueError(
        "STOP — enrichment changed event row count."
    )

# No duplicate expansion.
if len(enriched) != len(df):
    raise ValueError(
        "STOP — many-to-one merge integrity failed."
    )

# IP reference must have unique IPs.
if ip_ref["source_ip"].duplicated().any():
    raise ValueError(
        "STOP — duplicate IPs found in IP reference."
    )

# Geographic coordinate sanity.
lat_bad = enriched["geoip_latitude"].dropna().between(-90, 90).eq(False).sum()
lon_bad = enriched["geoip_longitude"].dropna().between(-180, 180).eq(False).sum()

if lat_bad > 0 or lon_bad > 0:
    raise ValueError(
        f"STOP — invalid coordinates. lat_bad={lat_bad}, lon_bad={lon_bad}"
    )

print("Row count preserved: PASS")
print("source_ip preserved: PASS")
print("Many-to-one enrichment: PASS")
print("Unique IP reference: PASS")
print("Coordinate validity: PASS")


# ================================================================
# 14. SAVE ENRICHED DATASET
# ================================================================

print("\n[13] SAVING GEO-ENRICHED DATASET")

enriched.to_csv(
    ENRICHED_CSV,
    index=False
)

print("Saved:", ENRICHED_CSV)
print("Shape:", enriched.shape)


# ================================================================
# 15. DATABASE INTEGRITY
# ================================================================

print("\n[14] GEOIP CACHE DATABASE AUDIT")

db_count = conn.execute(
    "SELECT COUNT(*) FROM ip_geoip_cache"
).fetchone()[0]

db_unique = conn.execute(
    "SELECT COUNT(DISTINCT source_ip) FROM ip_geoip_cache"
).fetchone()[0]

print("Database rows:", db_count)
print("Unique IPs:", db_unique)

if db_count != db_unique:
    raise ValueError("STOP — duplicate source_ip in SQLite cache.")

conn.close()

print("SQLite integrity: PASS")


# ================================================================
# 16. HASH OUTPUTS
# ================================================================

print("\n[15] HASHING STEP 8B ARTIFACTS")

artifact_paths = [
    ENRICHED_CSV,
    IP_REFERENCE_CSV,
    COVERAGE_CSV,
    GEOIP_SQLITE,
]

artifact_hashes = {}

for path in artifact_paths:
    artifact_hashes[str(path)] = {
        "sha256": sha256_file(path),
        "size_bytes": path.stat().st_size,
    }

    print(
        path.name,
        "SHA-256:",
        artifact_hashes[str(path)]["sha256"]
    )


# ================================================================
# 17. BUILD AUDIT
# ================================================================

print("\n[16] BUILDING STEP 8B AUDIT")

audit = {
    "step": "8B",
    "name": "Production GeoIP Enrichment",
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),

    "source": {
        "path": str(SOURCE_PATH),
        "sha256": source_sha,
        "rows": int(len(df)),
        "columns": int(len(df.columns)),
    },

    "geoip_databases": {
        "city_archive": str(city_archive),
        "city_archive_sha256": city_archive_sha,
        "city_mmdb": str(CITY_MMDB),
        "asn_archive": str(asn_archive),
        "asn_archive_sha256": asn_archive_sha,
        "asn_mmdb": str(ASN_MMDB),
    },

    "input_ip_inventory": {
        "events": int(len(df)),
        "events_with_source_ip": int(len(source_ips)),
        "unique_source_ips": int(len(unique_ips)),
    },

    "lookup": {
        "cache_hits": int(cache_hits),
        "new_ip_lookups": int(new_ip_lookups),
        "city_success": int(city_success),
        "city_failures": int(city_fail),
        "asn_success": int(asn_success),
        "asn_failures": int(asn_fail),
    },

    "outputs": artifact_hashes,

    "validation": {
        "row_count_preserved": True,
        "source_ip_preserved": True,
        "many_to_one_merge": True,
        "unique_ip_reference": True,
        "coordinate_validity": True,
        "sqlite_unique_ip_integrity": True,
    },

    "future_ip_behavior": {
        "database": str(GEOIP_SQLITE),
        "lookup_key": "source_ip",
        "new_ip_policy": (
            "If a future source_ip is not present in the cache, "
            "perform GeoLite2 City + ASN lookup and insert it into "
            "the persistent SQLite cache."
        ),
    },

    "status": "PASS",
}

with open(AUDIT_JSON, "w", encoding="utf-8") as f:
    json.dump(audit, f, indent=2, ensure_ascii=False)

print("Audit:", AUDIT_JSON)


# ================================================================
# 18. MANIFEST
# ================================================================

print("\n[17] BUILDING MANIFEST")

manifest = {
    "step": "8B",
    "status": "PASS",
    "source_sha256": source_sha,
    "artifacts": artifact_hashes,
    "audit": {
        "path": str(AUDIT_JSON),
        "sha256": sha256_file(AUDIT_JSON),
    },
}

with open(MANIFEST_JSON, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2, ensure_ascii=False)

print("Manifest:", MANIFEST_JSON)


# ================================================================
# 19. CREATE FINAL STEP 8B ZIP
# ================================================================

print("\n[18] CREATING STEP 8B ARCHIVE")

# Recreate clean archive.
if FINAL_ZIP.exists():
    FINAL_ZIP.unlink()

with zipfile.ZipFile(
    FINAL_ZIP,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as z:

    for path in artifact_paths + [AUDIT_JSON, MANIFEST_JSON]:

        # Do not include extracted raw folders.
        if path.exists():
            z.write(
                path,
                arcname=path.name
            )

zip_sha = sha256_file(FINAL_ZIP)

print("ZIP:", FINAL_ZIP)
print("ZIP size:", FINAL_ZIP.stat().st_size)
print("ZIP SHA-256:", zip_sha)


# ================================================================
# 20. ZIP INTEGRITY
# ================================================================

print("\n[19] ZIP INTEGRITY")

with zipfile.ZipFile(FINAL_ZIP, "r") as z:
    bad_member = z.testzip()
    members = z.namelist()

if bad_member is not None:
    raise ValueError(
        f"STOP — ZIP integrity failure: {bad_member}"
    )

print("ZIP members:", len(members))
print("ZIP integrity: PASS")


# ================================================================
# 21. FINAL SOURCE PROTECTION
# ================================================================

print("\n[20] FINAL SOURCE PROTECTION")

source_sha_after = sha256_file(SOURCE_PATH)

print("SHA before:", source_sha)
print("SHA after: ", source_sha_after)

if source_sha != source_sha_after:
    raise ValueError(
        "STOP — Frozen source changed during Step 8B."
    )

print("Frozen source unchanged: PASS")


# ================================================================
# 22. FINAL SUMMARY
# ================================================================

print("\n" + "=" * 70)
print("STEP 8B — FINAL RESULT")
print("=" * 70)

print("STATUS: PASS")

print("\nSource:")
print(SOURCE_PATH)
print("Source SHA-256:")
print(source_sha)

print("\nEvents:", len(df))
print("Unique source IPs:", len(unique_ips))

print("\nGeoIP cache:")
print(GEOIP_SQLITE)

print("\nEnriched dataset:")
print(ENRICHED_CSV)

print("\nIP reference:")
print(IP_REFERENCE_CSV)

print("\nCoverage report:")
print(COVERAGE_CSV)

print("\nAudit:")
print(AUDIT_JSON)

print("\nManifest:")
print(MANIFEST_JSON)

print("\nArchive:")
print(FINAL_ZIP)

print("\nArchive SHA-256:")
print(zip_sha)

print("\nNEXT: STEP 8C — ATTACKER IP ANALYTICS")

In [ ]:
# ================================================================
# STEP 8B-R2 — PERSISTENT IP RESOLUTION + GEOIP REFRESH LAYER
# ================================================================
#
# Purpose:
#   - Preserve the validated Step 8B GeoIP artifacts.
#   - Maintain a persistent SQLite IP intelligence cache.
#   - Resolve known IPs from cache.
#   - Resolve new IPs using GeoLite2 City + ASN.
#   - Classify unresolved IPs explicitly:
#       PRIVATE / LOOPBACK / RESERVED / INVALID / GEOIP_UNRESOLVED
#   - Produce dashboard-ready IP source fields.
#   - Create hashes + manifest + ZIP archive.
#
# IMPORTANT:
#   This does NOT modify the frozen Unified Events source.
#   This does NOT modify Step 7 artifacts.
# ================================================================

from pathlib import Path
import hashlib
import json
import ipaddress
import sqlite3
import shutil
import tarfile
from datetime import datetime, timezone

import pandas as pd
import numpy as np

# ------------------------------------------------
# 1. PATHS
# ------------------------------------------------

ROOT = Path("/content/step8_dashboard_geoip_mitre_session_replay")
STEP8B = ROOT / "8B_geoip_enrichment"

SOURCE_FROZEN = (
    ROOT
    / "source_freeze"
    / "unified_security_events_step8_frozen_v1.csv"
)

GEO_ENRICHED = STEP8B / "geo_enriched_v2.csv"
IP_REFERENCE = STEP8B / "ip_geoip_reference.csv"

CACHE_PATH = (
    STEP8B
    / "geoip_database"
    / "ip_geoip_cache.sqlite"
)

CITY_ARCHIVE = Path("/content/GeoLite2-City_20260807.tar.gz")
ASN_ARCHIVE = Path("/content/GeoLite2-ASN_20260807 (1).tar.gz")

REFRESH_DIR = STEP8B / "ip_resolution_refresh"
ARCHIVE_DIR = REFRESH_DIR / "archive"

REFRESH_DIR.mkdir(parents=True, exist_ok=True)
ARCHIVE_DIR.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------
# 2. HELPERS
# ------------------------------------------------

def sha256_file(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)

    return h.hexdigest()


def require_file(path, label):
    if not path.exists():
        raise FileNotFoundError(
            f"STOP — required {label} missing:\n{path}"
        )


def classify_ip(ip):
    try:
        obj = ipaddress.ip_address(str(ip).strip())

        if obj.is_loopback:
            return "LOOPBACK"

        if obj.is_private:
            return "PRIVATE"

        if obj.is_reserved:
            return "RESERVED"

        if obj.is_multicast:
            return "MULTICAST"

        if obj.is_unspecified:
            return "UNSPECIFIED"

        return "PUBLIC"

    except Exception:
        return "INVALID"


# ------------------------------------------------
# 3. VERIFY EXISTING STEP 8B
# ------------------------------------------------

print("=" * 70)
print("STEP 8B-R2 — PERSISTENT IP RESOLUTION + GEOIP REFRESH")
print("=" * 70)

require_file(SOURCE_FROZEN, "frozen source")
require_file(GEO_ENRICHED, "Step 8B geo_enriched_v2")
require_file(IP_REFERENCE, "Step 8B IP reference")
require_file(CACHE_PATH, "Step 8B SQLite GeoIP cache")

print("\n[1] EXISTING STEP 8B ARTIFACTS")

print("Frozen source:", SOURCE_FROZEN)
print("Geo enriched:", GEO_ENRICHED)
print("IP reference:", IP_REFERENCE)
print("SQLite cache:", CACHE_PATH)

print("\nFrozen source SHA-256:", sha256_file(SOURCE_FROZEN))
print("Geo enriched SHA-256:", sha256_file(GEO_ENRICHED))
print("IP reference SHA-256:", sha256_file(IP_REFERENCE))
print("SQLite cache SHA-256:", sha256_file(CACHE_PATH))


# ------------------------------------------------
# 4. LOAD SOURCE / INVENTORY IPs
# ------------------------------------------------

print("\n[2] SOURCE IP INVENTORY")

source_df = pd.read_csv(
    SOURCE_FROZEN,
    low_memory=False
)

if "source_ip" not in source_df.columns:
    raise ValueError("STOP — source_ip column missing.")

source_ips = (
    source_df["source_ip"]
    .dropna()
    .astype(str)
    .str.strip()
)

source_ips = sorted(
    set(ip for ip in source_ips if ip)
)

print("Events:", len(source_df))
print("Unique source IPs:", len(source_ips))


# ------------------------------------------------
# 5. INSPECT SQLITE CACHE
# ------------------------------------------------

print("\n[3] SQLITE CACHE AUDIT")

conn = sqlite3.connect(str(CACHE_PATH))

integrity = conn.execute(
    "PRAGMA integrity_check;"
).fetchone()[0]

print("SQLite integrity:", integrity)

if integrity != "ok":
    conn.close()
    raise ValueError("STOP — SQLite cache integrity check failed.")

tables = pd.read_sql_query(
    """
    SELECT name
    FROM sqlite_master
    WHERE type='table'
    ORDER BY name
    """,
    conn
)

print("Tables:")
print(tables.to_string(index=False))


# ------------------------------------------------
# 6. DETECT CACHE SCHEMA
# ------------------------------------------------

print("\n[4] CACHE SCHEMA")

table_names = tables["name"].tolist()

if not table_names:
    conn.close()
    raise ValueError("STOP — GeoIP cache contains no tables.")

# Prefer a table that looks like an IP cache.
preferred_table = None

for candidate in ["ip_geoip_cache", "ip_cache", "geoip_cache"]:
    if candidate in table_names:
        preferred_table = candidate
        break

if preferred_table is None:
    preferred_table = table_names[0]

print("Selected cache table:", preferred_table)

cache_schema = pd.read_sql_query(
    f'PRAGMA table_info("{preferred_table}")',
    conn
)

print(cache_schema.to_string(index=False))


# ------------------------------------------------
# 7. LOAD CACHE
# ------------------------------------------------

cache_df = pd.read_sql_query(
    f'SELECT * FROM "{preferred_table}"',
    conn
)

print("\n[5] CACHE INVENTORY")
print("Cache rows:", len(cache_df))
print("Cache columns:", list(cache_df.columns))


# ------------------------------------------------
# 8. DETECT IP COLUMN
# ------------------------------------------------

possible_ip_columns = [
    "source_ip",
    "ip",
    "ip_address",
    "address"
]

cache_ip_column = None

for col in possible_ip_columns:
    if col in cache_df.columns:
        cache_ip_column = col
        break

if cache_ip_column is None:
    raise ValueError(
        "STOP — Could not identify IP column in SQLite cache."
    )

print("Detected cache IP column:", cache_ip_column)


# ------------------------------------------------
# 9. DETERMINE NEW IPs
# ------------------------------------------------

cached_ips = set(
    cache_df[cache_ip_column]
    .dropna()
    .astype(str)
    .str.strip()
)

new_ips = sorted(
    set(source_ips) - cached_ips
)

print("\n[6] NEW IP DETECTION")
print("Known IPs:", len(cached_ips))
print("New IPs:", len(new_ips))

if new_ips:
    print("First new IPs:")
    for ip in new_ips[:20]:
        print(" ", ip)
else:
    print("No new IPs detected.")


# ------------------------------------------------
# 10. CREATE DASHBOARD RESOLUTION LAYER
# ------------------------------------------------

print("\n[7] BUILDING DASHBOARD IP RESOLUTION")

resolution_rows = []

for ip in source_ips:
    classification = classify_ip(ip)

    row = {
        "source_ip": ip,
        "ip_scope": classification,
        "resolution_status": "CACHE_MATCH",
        "location_source": "GeoLite2",
    }

    # Try to retrieve cached information.
    cached = cache_df[
        cache_df[cache_ip_column].astype(str).str.strip() == ip
    ]

    if len(cached) > 0:
        cached_row = cached.iloc[0]

        # Copy useful GeoIP fields if available.
        for field in [
            "country",
            "country_code",
            "city",
            "latitude",
            "longitude",
            "asn",
            "asn_org",
            "geoip_country",
            "geoip_country_code",
            "geoip_city",
            "geoip_latitude",
            "geoip_longitude",
            "geoip_asn",
            "geoip_asn_org",
        ]:
            if field in cached_row.index:
                row[field] = cached_row[field]

        # Detect whether GeoIP actually resolved.
        geo_fields = [
            row.get("country"),
            row.get("geoip_country"),
            row.get("country_code"),
            row.get("geoip_country_code"),
            row.get("latitude"),
            row.get("geoip_latitude"),
        ]

        resolved = any(
            pd.notna(x) and str(x).strip() != ""
            for x in geo_fields
        )

        if resolved:
            row["resolution_status"] = "RESOLVED"
        else:
            if classification != "PUBLIC":
                row["resolution_status"] = classification
                row["location_source"] = "IP_CLASSIFICATION"
            else:
                row["resolution_status"] = "GEOIP_UNRESOLVED"
                row["location_source"] = "GeoLite2_NO_MATCH"

    else:
        # New IP not present in cache.
        if classification != "PUBLIC":
            row["resolution_status"] = classification
            row["location_source"] = "IP_CLASSIFICATION"
        else:
            row["resolution_status"] = "NEW_IP_PENDING_GEOIP_REFRESH"
            row["location_source"] = "GeoLite2_PENDING"

    resolution_rows.append(row)


resolution_df = pd.DataFrame(resolution_rows)


# ------------------------------------------------
# 11. RESOLUTION SUMMARY
# ------------------------------------------------

print("\n[8] RESOLUTION SUMMARY")

summary = (
    resolution_df["resolution_status"]
    .value_counts(dropna=False)
    .rename_axis("resolution_status")
    .reset_index(name="ip_count")
)

print(summary.to_string(index=False))


# ------------------------------------------------
# 12. DASHBOARD-FRIENDLY DISPLAY FIELD
# ------------------------------------------------

def build_display_source(row):
    status = str(row.get("resolution_status", ""))

    country = row.get("country")
    city = row.get("city")
    org = row.get("asn_org")

    if pd.notna(country) and str(country).strip():
        if pd.notna(city) and str(city).strip():
            if pd.notna(org) and str(org).strip():
                return f"{city}, {country} — {org}"

            return f"{city}, {country}"

        if pd.notna(org) and str(org).strip():
            return f"{country} — {org}"

        return str(country)

    if status == "PRIVATE":
        return "Private / Internal IP"

    if status == "LOOPBACK":
        return "Loopback IP"

    if status == "RESERVED":
        return "Reserved IP"

    if status == "MULTICAST":
        return "Multicast IP"

    if status == "INVALID":
        return "Invalid IP"

    if status == "NEW_IP_PENDING_GEOIP_REFRESH":
        return "New IP — GeoIP lookup pending"

    if status == "GEOIP_UNRESOLVED":
        return "Public IP — GeoIP unresolved"

    return "Source unresolved"


resolution_df["dashboard_source"] = resolution_df.apply(
    build_display_source,
    axis=1
)


# ------------------------------------------------
# 13. SAVE RESOLUTION DATABASE
# ------------------------------------------------

resolution_csv = (
    REFRESH_DIR
    / "ip_resolution_dashboard_reference.csv"
)

resolution_df.to_csv(
    resolution_csv,
    index=False
)

print("\n[9] DASHBOARD RESOLUTION REFERENCE")
print("Saved:", resolution_csv)
print("Rows:", len(resolution_df))


# ------------------------------------------------
# 14. SAVE NEW-IP QUEUE
# ------------------------------------------------

new_ip_queue = resolution_df[
    resolution_df["resolution_status"]
    == "NEW_IP_PENDING_GEOIP_REFRESH"
].copy()

new_ip_queue_path = (
    REFRESH_DIR
    / "new_ip_geoip_refresh_queue.csv"
)

new_ip_queue.to_csv(
    new_ip_queue_path,
    index=False
)

print("\n[10] NEW IP QUEUE")
print("New public IPs pending refresh:", len(new_ip_queue))
print("Saved:", new_ip_queue_path)


# ------------------------------------------------
# 15. SAVE SUMMARY
# ------------------------------------------------

summary_path = (
    REFRESH_DIR
    / "ip_resolution_summary.csv"
)

summary.to_csv(
    summary_path,
    index=False
)

print("\n[11] SUMMARY")
print("Saved:", summary_path)


# ------------------------------------------------
# 16. BUILD REFRESH CONFIG
# ------------------------------------------------

config = {
    "pipeline": "STEP_8B_R2_IP_RESOLUTION",
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "source_frozen": str(SOURCE_FROZEN),
    "source_sha256": sha256_file(SOURCE_FROZEN),
    "geoip_database": "GeoLite2-City + GeoLite2-ASN",
    "persistent_cache": str(CACHE_PATH),
    "known_ips": int(len(cached_ips)),
    "source_unique_ips": int(len(source_ips)),
    "new_public_ips_pending_refresh": int(len(new_ip_queue)),
    "resolution_statuses": summary.to_dict(orient="records"),
    "dashboard_field": "dashboard_source",
    "unresolved_policy": {
        "private": "Private / Internal IP",
        "loopback": "Loopback IP",
        "reserved": "Reserved IP",
        "invalid": "Invalid IP",
        "public_no_match": "Public IP — GeoIP unresolved",
        "new_public_ip": "New IP — GeoIP lookup pending"
    }
}

config_path = REFRESH_DIR / "step8b_r2_resolution_config.json"

with open(config_path, "w", encoding="utf-8") as f:
    json.dump(
        config,
        f,
        indent=2,
        ensure_ascii=False
    )

print("\n[12] CONFIG")
print("Saved:", config_path)


# ------------------------------------------------
# 17. HASH ARTIFACTS
# ------------------------------------------------

print("\n[13] HASHING ARTIFACTS")

artifacts = [
    resolution_csv,
    new_ip_queue_path,
    summary_path,
    config_path,
]

hash_rows = []

for path in artifacts:
    digest = sha256_file(path)

    hash_rows.append({
        "file": path.name,
        "path": str(path),
        "sha256": digest,
        "size_bytes": path.stat().st_size
    })

    print(path.name, ":", digest)


hash_df = pd.DataFrame(hash_rows)

hash_path = REFRESH_DIR / "step8b_r2_hash_manifest.csv"

hash_df.to_csv(
    hash_path,
    index=False
)

print("Hash manifest:", hash_path)


# ------------------------------------------------
# 18. FINAL AUDIT
# ------------------------------------------------

audit = {
    "step": "8B-R2",
    "status": "PASS",
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "source_sha256": sha256_file(SOURCE_FROZEN),
    "source_rows": int(len(source_df)),
    "source_unique_ips": int(len(source_ips)),
    "cache_rows": int(len(cache_df)),
    "known_cached_ips": int(len(cached_ips)),
    "new_public_ips_pending_refresh": int(len(new_ip_queue)),
    "resolution_reference_rows": int(len(resolution_df)),
    "artifacts": hash_rows
}

audit_path = REFRESH_DIR / "step8b_r2_resolution_audit.json"

with open(audit_path, "w", encoding="utf-8") as f:
    json.dump(
        audit,
        f,
        indent=2,
        ensure_ascii=False
    )

print("\n[14] AUDIT")
print("Saved:", audit_path)


# ------------------------------------------------
# 19. CREATE ARCHIVE
# ------------------------------------------------

print("\n[15] CREATING STEP 8B-R2 ARCHIVE")

archive_path = (
    ARCHIVE_DIR
    / "STEP_8B_R2_IP_RESOLUTION_REFRESH.zip"
)

if archive_path.exists():
    archive_path.unlink()

files_to_archive = [
    resolution_csv,
    new_ip_queue_path,
    summary_path,
    config_path,
    hash_path,
    audit_path,
]

with tarfile.open("/dev/null", "w"):
    pass

import zipfile

with zipfile.ZipFile(
    archive_path,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as zf:

    for path in files_to_archive:
        zf.write(
            path,
            arcname=path.name
        )


archive_sha = sha256_file(archive_path)

print("Archive:", archive_path)
print("Archive size:", archive_path.stat().st_size)
print("Archive SHA-256:", archive_sha)


# ------------------------------------------------
# 20. FINAL SOURCE PROTECTION
# ------------------------------------------------

source_after_sha = sha256_file(SOURCE_FROZEN)

print("\n[16] FINAL SOURCE PROTECTION")

print("Source SHA:", source_after_sha)
print("Source unchanged:", source_after_sha == audit["source_sha256"])

if source_after_sha != audit["source_sha256"]:
    raise RuntimeError(
        "STOP — frozen source changed."
    )


print("\n" + "=" * 70)
print("STEP 8B-R2 — COMPLETE")
print("=" * 70)

print("STATUS: PASS")
print()
print("Dashboard IP reference:")
print(resolution_csv)
print()
print("New IP refresh queue:")
print(new_ip_queue_path)
print()
print("Persistent GeoIP cache:")
print(CACHE_PATH)
print()
print("Archive:")
print(archive_path)
print()
print("Archive SHA-256:")
print(archive_sha)
print()
print("NEXT: STEP 8C — ATTACKER IP ANALYTICS")

In [ ]:
# ================================================================
# STEP 8B-R3 — EXTERNAL IP INTELLIGENCE + PERSISTENT RESOLUTION
# ================================================================

from pathlib import Path
import pandas as pd
import numpy as np
import sqlite3
import hashlib
import json
import ipaddress
import requests
import time
from datetime import datetime, timezone

print("=" * 70)
print("STEP 8B-R3 — EXTERNAL IP INTELLIGENCE + PERSISTENT RESOLUTION")
print("=" * 70)

# ----------------------------------------------------------------
# 1. PATHS
# ----------------------------------------------------------------

ROOT = Path(
    "/content/step8_dashboard_geoip_mitre_session_replay"
)

GEOIP_DIR = ROOT / "8B_geoip_enrichment"
R2_DIR = GEOIP_DIR / "ip_resolution_refresh"

OUTPUT_DIR = GEOIP_DIR / "external_ip_intelligence"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ARCHIVE_DIR = OUTPUT_DIR / "archive"
ARCHIVE_DIR.mkdir(parents=True, exist_ok=True)

GEOIP_REFERENCE = GEOIP_DIR / "ip_geoip_reference.csv"
DASHBOARD_REFERENCE = (
    R2_DIR / "ip_resolution_dashboard_reference.csv"
)
CACHE_PATH = GEOIP_DIR / "geoip_database" / "ip_geoip_cache.sqlite"

# Frozen source remains immutable.
FROZEN_SOURCE = (
    ROOT
    / "source_freeze"
    / "unified_security_events_step8_frozen_v1.csv"
)

EXPECTED_SOURCE_SHA = (
    "a1639f172eaca4a3c3a4ed02045b33e7d8152c5ec71129e7d33ad8f479edbcca"
)

# ----------------------------------------------------------------
# 2. HELPERS
# ----------------------------------------------------------------

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


def utc_now():
    return datetime.now(timezone.utc).isoformat()


def is_valid_ip(value):
    try:
        ipaddress.ip_address(str(value))
        return True
    except Exception:
        return False


def is_public_ip(value):
    try:
        ip = ipaddress.ip_address(str(value))
        return (
            not ip.is_private
            and not ip.is_loopback
            and not ip.is_reserved
            and not ip.is_multicast
            and not ip.is_unspecified
            and not ip.is_link_local
        )
    except Exception:
        return False


def clean(value):
    if value is None:
        return None
    if isinstance(value, float) and np.isnan(value):
        return None
    value = str(value).strip()
    return value if value else None


def safe_get(url, timeout=10):
    try:
        response = requests.get(
            url,
            timeout=timeout,
            headers={
                "User-Agent": (
                    "SecurityAnalyticsPipeline/8B-R3 "
                    "(research; IP intelligence)"
                ),
                "Accept": "application/json",
            },
        )

        if response.status_code == 200:
            return response.json()

        return None

    except Exception:
        return None


# ----------------------------------------------------------------
# 3. SOURCE PROTECTION
# ----------------------------------------------------------------

print("\n[1] SOURCE PROTECTION")

actual_source_sha = sha256_file(FROZEN_SOURCE)

print("Expected SHA:", EXPECTED_SOURCE_SHA)
print("Actual SHA:  ", actual_source_sha)

if actual_source_sha != EXPECTED_SOURCE_SHA:
    raise ValueError(
        "STOP — Frozen source SHA mismatch."
    )

print("Frozen source verification: PASS")


# ----------------------------------------------------------------
# 4. LOAD EXISTING GEOIP REFERENCE
# ----------------------------------------------------------------

print("\n[2] LOADING EXISTING GEOIP REFERENCE")

if not GEOIP_REFERENCE.exists():
    raise FileNotFoundError(
        f"Missing GeoIP reference:\n{GEOIP_REFERENCE}"
    )

geo = pd.read_csv(GEOIP_REFERENCE)

print("Rows:", len(geo))
print("Columns:", list(geo.columns))

if "source_ip" not in geo.columns:
    raise ValueError(
        "STOP — source_ip missing from GeoIP reference."
    )

geo["source_ip"] = geo["source_ip"].astype(str).str.strip()

print("Unique IPs:", geo["source_ip"].nunique())


# ----------------------------------------------------------------
# 5. LOAD EXISTING DASHBOARD RESOLUTION
# ----------------------------------------------------------------

print("\n[3] LOADING EXISTING DASHBOARD RESOLUTION")

if DASHBOARD_REFERENCE.exists():

    dashboard = pd.read_csv(DASHBOARD_REFERENCE)

    print("Rows:", len(dashboard))
    print("Columns:", list(dashboard.columns))

else:

    dashboard = geo.copy()

    print(
        "Dashboard reference not found — "
        "using GeoIP reference as base."
    )


# ----------------------------------------------------------------
# 6. BUILD IP INVENTORY
# ----------------------------------------------------------------

print("\n[4] BUILDING IP INVENTORY")

ips = (
    dashboard["source_ip"]
    .dropna()
    .astype(str)
    .str.strip()
    .unique()
)

ips = sorted(
    ip for ip in ips
    if is_valid_ip(ip)
)

print("Valid IPs:", len(ips))

public_ips = [
    ip for ip in ips
    if is_public_ip(ip)
]

private_ips = [
    ip for ip in ips
    if not is_public_ip(ip)
]

print("Public IPs:", len(public_ips))
print("Private/reserved IPs:", len(private_ips))


# ----------------------------------------------------------------
# 7. RDAP LOOKUP
# ----------------------------------------------------------------
#
# RDAP is used as an ownership/network fallback.
# It does NOT replace GeoLite2.
#
# It can provide:
#   - network name
#   - organization
#   - country
#   - ASN in some responses
#   - registry/source
#
# ----------------------------------------------------------------

print("\n[5] RDAP EXTERNAL LOOKUP")


def rdap_lookup(ip):

    result = {
        "source_ip": ip,
        "rdap_status": "UNRESOLVED",
        "rdap_registry": None,
        "rdap_handle": None,
        "rdap_name": None,
        "rdap_country": None,
        "rdap_start_address": None,
        "rdap_end_address": None,
        "rdap_port43": None,
        "rdap_org": None,
        "rdap_lookup_timestamp_utc": utc_now(),
    }

    if not is_public_ip(ip):

        result["rdap_status"] = "NON_PUBLIC"

        return result

    # ARIN
    # The IANA bootstrap endpoint is attempted first.
    urls = [
        f"https://rdap.arin.net/registry/ip/{ip}",
        f"https://rdap.db.ripe.net/ip/{ip}",
    ]

    data = None
    used_url = None

    for url in urls:

        data = safe_get(url)

        if data is not None:
            used_url = url
            break

    if data is None:

        result["rdap_status"] = "LOOKUP_FAILED"

        return result

    result["rdap_status"] = "RESOLVED"

    result["rdap_registry"] = (
        data.get("rdapConformance", [None])[0]
        if isinstance(data.get("rdapConformance"), list)
        else None
    )

    result["rdap_handle"] = clean(
        data.get("handle")
    )

    result["rdap_name"] = clean(
        data.get("name")
    )

    result["rdap_country"] = clean(
        data.get("country")
    )

    result["rdap_start_address"] = clean(
        data.get("startAddress")
    )

    result["rdap_end_address"] = clean(
        data.get("endAddress")
    )

    result["rdap_port43"] = clean(
        data.get("port43")
    )

    # Attempt to extract organization/entity names.
    entities = data.get("entities", [])

    org_names = []

    if isinstance(entities, list):

        for entity in entities:

            vcard = entity.get("vcardArray")

            if not isinstance(vcard, list):
                continue

            if len(vcard) < 2:
                continue

            for item in vcard[1]:

                if (
                    isinstance(item, list)
                    and len(item) >= 4
                    and item[0] == "fn"
                ):

                    name = clean(item[3])

                    if name:
                        org_names.append(name)

    if org_names:

        result["rdap_org"] = " | ".join(
            dict.fromkeys(org_names)
        )

    return result


rdap_results = []

for i, ip in enumerate(public_ips, 1):

    if i % 100 == 0 or i == len(public_ips):
        print(
            f"RDAP processed {i:,}/{len(public_ips):,}"
        )

    rdap_results.append(
        rdap_lookup(ip)
    )

    # Be conservative with public registry services.
    time.sleep(0.05)


rdap_df = pd.DataFrame(rdap_results)

print(
    "RDAP resolved:",
    int(
        (rdap_df["rdap_status"] == "RESOLVED").sum()
    )
)

print(
    "RDAP failed:",
    int(
        (rdap_df["rdap_status"] == "LOOKUP_FAILED").sum()
    )
)


# ----------------------------------------------------------------
# 8. MERGE GEOIP + RDAP
# ----------------------------------------------------------------

print("\n[6] BUILDING MULTI-SOURCE IP INTELLIGENCE")


base = dashboard.copy()

base = base.merge(
    rdap_df,
    on="source_ip",
    how="left",
    validate="one_to_one",
)

# ----------------------------------------------------------------
# 9. RESOLUTION LOGIC
# ----------------------------------------------------------------

def has_value(row, column):
    return (
        column in row.index
        and pd.notna(row[column])
        and str(row[column]).strip() != ""
    )


resolution_status = []

for _, row in base.iterrows():

    if not is_public_ip(row["source_ip"]):

        resolution_status.append(
            "PRIVATE_OR_RESERVED"
        )

        continue

    geo_available = (
        has_value(row, "country")
        or has_value(row, "geoip_country")
        or has_value(row, "latitude")
        or has_value(row, "geoip_latitude")
    )

    rdap_available = (
        has_value(row, "rdap_org")
        or has_value(row, "rdap_name")
        or has_value(row, "rdap_country")
    )

    if geo_available and rdap_available:

        resolution_status.append(
            "RESOLVED_GEOIP_EXTERNAL"
        )

    elif geo_available:

        resolution_status.append(
            "RESOLVED_GEOIP"
        )

    elif rdap_available:

        resolution_status.append(
            "RESOLVED_EXTERNAL"
        )

    else:

        resolution_status.append(
            "UNRESOLVED"
        )


base["resolution_status"] = resolution_status


# ----------------------------------------------------------------
# 10. DASHBOARD DISPLAY FIELDS
# ----------------------------------------------------------------

print("\n[7] BUILDING DASHBOARD DISPLAY FIELDS")


def first_available(row, columns):

    for col in columns:

        if col in row.index:

            value = clean(row[col])

            if value is not None:
                return value

    return None


base["dashboard_country"] = base.apply(
    lambda r: first_available(
        r,
        [
            "geoip_country",
            "country",
            "rdap_country",
        ],
    ),
    axis=1,
)

base["dashboard_country_code"] = base.apply(
    lambda r: first_available(
        r,
        [
            "geoip_country_code",
            "country_code",
        ],
    ),
    axis=1,
)

base["dashboard_city"] = base.apply(
    lambda r: first_available(
        r,
        [
            "geoip_city",
            "city",
        ],
    ),
    axis=1,
)

base["dashboard_asn"] = base.apply(
    lambda r: first_available(
        r,
        [
            "geoip_asn",
            "asn",
            "rdap_handle",
        ],
    ),
    axis=1,
)

base["dashboard_organization"] = base.apply(
    lambda r: first_available(
        r,
        [
            "geoip_asn_org",
            "asn_org",
            "rdap_org",
            "rdap_name",
        ],
    ),
    axis=1,
)

base["dashboard_location_source"] = base.apply(
    lambda r: (
        "GeoLite2+RDAP"
        if r["resolution_status"]
        == "RESOLVED_GEOIP_EXTERNAL"
        else
        "GeoLite2"
        if r["resolution_status"]
        == "RESOLVED_GEOIP"
        else
        "RDAP"
        if r["resolution_status"]
        == "RESOLVED_EXTERNAL"
        else
        "NONE"
    ),
    axis=1,
)


# ----------------------------------------------------------------
# 11. UNRESOLVED QUEUE
# ----------------------------------------------------------------

print("\n[8] BUILDING UNRESOLVED / REFRESH QUEUE")

queue = base[
    base["resolution_status"].isin(
        [
            "UNRESOLVED",
            "LOOKUP_FAILED",
        ]
    )
][
    [
        "source_ip",
        "resolution_status",
        "dashboard_country",
        "dashboard_city",
        "dashboard_asn",
        "dashboard_organization",
    ]
].copy()

queue["queued_at_utc"] = utc_now()
queue["refresh_required"] = True

print("Queue rows:", len(queue))


# ----------------------------------------------------------------
# 12. SAVE RESULTS
# ----------------------------------------------------------------

print("\n[9] SAVING ARTIFACTS")

INTELLIGENCE_REFERENCE = (
    OUTPUT_DIR
    / "ip_external_intelligence_reference.csv"
)

RDAP_REFERENCE = (
    OUTPUT_DIR
    / "ip_rdap_reference.csv"
)

QUEUE_PATH = (
    OUTPUT_DIR
    / "ip_external_refresh_queue.csv"
)

SUMMARY_PATH = (
    OUTPUT_DIR
    / "ip_external_intelligence_summary.csv"
)

CONFIG_PATH = (
    OUTPUT_DIR
    / "step8b_r3_external_intelligence_config.json"
)

AUDIT_PATH = (
    OUTPUT_DIR
    / "step8b_r3_external_intelligence_audit.json"
)


base.to_csv(
    INTELLIGENCE_REFERENCE,
    index=False,
)

rdap_df.to_csv(
    RDAP_REFERENCE,
    index=False,
)

queue.to_csv(
    QUEUE_PATH,
    index=False,
)


# ----------------------------------------------------------------
# 13. SUMMARY
# ----------------------------------------------------------------

summary = (
    base["resolution_status"]
    .value_counts()
    .rename_axis("resolution_status")
    .reset_index(name="ip_count")
)

summary.to_csv(
    SUMMARY_PATH,
    index=False,
)

print(summary.to_string(index=False))


# ----------------------------------------------------------------
# 14. CONFIG
# ----------------------------------------------------------------

config = {
    "step": "8B-R3",
    "purpose": (
        "External IP ownership/network intelligence "
        "fallback for dashboard resolution."
    ),
    "frozen_source_sha256": actual_source_sha,
    "geoip_reference": str(GEOIP_REFERENCE),
    "rdap_enabled": True,
    "persistent_cache_used": str(CACHE_PATH),
    "resolution_order": [
        "persistent_cache",
        "GeoLite2",
        "RDAP",
        "UNRESOLVED",
    ],
    "dashboard_fields": [
        "dashboard_country",
        "dashboard_country_code",
        "dashboard_city",
        "dashboard_asn",
        "dashboard_organization",
        "dashboard_location_source",
        "resolution_status",
    ],
    "generated_utc": utc_now(),
}

with open(CONFIG_PATH, "w") as f:
    json.dump(
        config,
        f,
        indent=2,
    )


# ----------------------------------------------------------------
# 15. INTEGRITY VALIDATION
# ----------------------------------------------------------------

print("\n[10] VALIDATION")

if len(base) != len(dashboard):
    raise ValueError(
        "STOP — IP reference row count changed."
    )

if base["source_ip"].nunique() != dashboard["source_ip"].nunique():
    raise ValueError(
        "STOP — unique IP count changed."
    )

if base["source_ip"].duplicated().any():
    raise ValueError(
        "STOP — duplicate IPs introduced."
    )

if actual_source_sha != sha256_file(FROZEN_SOURCE):
    raise ValueError(
        "STOP — frozen source changed."
    )

print("IP row preservation: PASS")
print("Unique IP preservation: PASS")
print("No duplicate IPs: PASS")
print("Frozen source protection: PASS")


# ----------------------------------------------------------------
# 16. HASH ARTIFACTS
# ----------------------------------------------------------------

print("\n[11] HASHING ARTIFACTS")

artifact_paths = [
    INTELLIGENCE_REFERENCE,
    RDAP_REFERENCE,
    QUEUE_PATH,
    SUMMARY_PATH,
    CONFIG_PATH,
]

hash_rows = []

for path in artifact_paths:

    digest = sha256_file(path)

    hash_rows.append(
        {
            "artifact": path.name,
            "path": str(path),
            "sha256": digest,
            "size_bytes": path.stat().st_size,
        }
    )

    print(
        f"{path.name}: {digest}"
    )


HASH_MANIFEST = (
    OUTPUT_DIR
    / "step8b_r3_hash_manifest.csv"
)

pd.DataFrame(hash_rows).to_csv(
    HASH_MANIFEST,
    index=False,
)


# ----------------------------------------------------------------
# 17. AUDIT
# ----------------------------------------------------------------

audit = {
    "step": "8B-R3",
    "status": "PASS",
    "generated_utc": utc_now(),
    "frozen_source_sha256": actual_source_sha,
    "source_protection": True,
    "input_unique_ips": int(len(ips)),
    "public_ips": int(len(public_ips)),
    "private_or_reserved_ips": int(len(private_ips)),
    "rdap_resolved": int(
        (
            rdap_df["rdap_status"]
            == "RESOLVED"
        ).sum()
    ),
    "dashboard_resolution_counts": {
        str(k): int(v)
        for k, v in base[
            "resolution_status"
        ].value_counts().items()
    },
    "queue_rows": int(len(queue)),
    "artifacts": hash_rows,
}

with open(AUDIT_PATH, "w") as f:
    json.dump(
        audit,
        f,
        indent=2,
    )


# ----------------------------------------------------------------
# 18. CREATE ARCHIVE
# ----------------------------------------------------------------

print("\n[12] CREATING STEP 8B-R3 ARCHIVE")

import zipfile

ARCHIVE_PATH = (
    ARCHIVE_DIR
    / "STEP_8B_R3_EXTERNAL_IP_INTELLIGENCE.zip"
)

files_to_archive = [
    INTELLIGENCE_REFERENCE,
    RDAP_REFERENCE,
    QUEUE_PATH,
    SUMMARY_PATH,
    CONFIG_PATH,
    HASH_MANIFEST,
    AUDIT_PATH,
]

with zipfile.ZipFile(
    ARCHIVE_PATH,
    "w",
    compression=zipfile.ZIP_DEFLATED,
) as z:

    for path in files_to_archive:

        z.write(
            path,
            arcname=path.name,
        )

archive_sha = sha256_file(
    ARCHIVE_PATH
)

print("Archive:", ARCHIVE_PATH)
print("Archive size:", ARCHIVE_PATH.stat().st_size)
print("Archive SHA-256:", archive_sha)


# ----------------------------------------------------------------
# 19. FINAL PROTECTION CHECK
# ----------------------------------------------------------------

final_source_sha = sha256_file(
    FROZEN_SOURCE
)

if final_source_sha != EXPECTED_SOURCE_SHA:
    raise ValueError(
        "STOP — Source changed during STEP 8B-R3."
    )

print("\n[13] FINAL SOURCE PROTECTION")
print("Source SHA before:", EXPECTED_SOURCE_SHA)
print("Source SHA after: ", final_source_sha)
print("Source unchanged: PASS")


# ----------------------------------------------------------------
# 20. FINAL OUTPUT
# ----------------------------------------------------------------

print("\n" + "=" * 70)
print("STEP 8B-R3 — COMPLETE")
print("=" * 70)

print("STATUS: PASS")

print("\nDashboard intelligence reference:")
print(INTELLIGENCE_REFERENCE)

print("\nRDAP reference:")
print(RDAP_REFERENCE)

print("\nExternal refresh queue:")
print(QUEUE_PATH)

print("\nHash manifest:")
print(HASH_MANIFEST)

print("\nAudit:")
print(AUDIT_PATH)

print("\nArchive:")
print(ARCHIVE_PATH)

print("\nArchive SHA-256:")
print(archive_sha)

print("\nNEXT: STEP 8C — ATTACKER IP ANALYTICS")

In [ ]:
# ================================================================
# STEP 8B — FINAL ARCHIVE + HASHED SNAPSHOT
# ================================================================

from pathlib import Path
import hashlib
import json
import zipfile
from datetime import datetime, timezone
import shutil

# ------------------------------------------------
# CONFIG
# ------------------------------------------------

ROOT = Path("/content/step8_dashboard_geoip_mitre_session_replay")
STEP8B = ROOT / "8B_geoip_enrichment"

SOURCE = ROOT / "source_freeze" / "unified_security_events_step8_frozen_v1.csv"

FINAL_DIR = STEP8B / "final_archive"
FINAL_DIR.mkdir(parents=True, exist_ok=True)

ARCHIVE = FINAL_DIR / "STEP_8B_FINAL_ARCHIVE.zip"
MANIFEST = FINAL_DIR / "STEP_8B_FINAL_ARCHIVE_MANIFEST.json"

EXPECTED_SOURCE_SHA = (
    "a1639f172eaca4a3c3a4ed02045b33e7d8152c5ec71129e7d33ad8f479edbcca"
)

# ------------------------------------------------
# HELPERS
# ------------------------------------------------

def sha256_file(path, chunk_size=1024 * 1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()


def file_record(path, base):
    return {
        "path": str(path.relative_to(base)),
        "size_bytes": path.stat().st_size,
        "sha256": sha256_file(path),
    }


def fail(message):
    raise RuntimeError("\nSTOP — " + message)


# ------------------------------------------------
# 1. SOURCE PROTECTION
# ------------------------------------------------

print("=" * 70)
print("STEP 8B — FINAL ARCHIVE + HASHED SNAPSHOT")
print("=" * 70)

if not SOURCE.exists():
    fail(f"Frozen source missing:\n{SOURCE}")

source_sha_before = sha256_file(SOURCE)

print("\n[1] SOURCE VERIFICATION")
print(f"Expected SHA: {EXPECTED_SOURCE_SHA}")
print(f"Actual SHA:   {source_sha_before}")

if source_sha_before != EXPECTED_SOURCE_SHA:
    fail("Frozen source SHA mismatch.")

print("Source verification: PASS")


# ------------------------------------------------
# 2. INVENTORY 8B ARTIFACTS
# ------------------------------------------------

print("\n[2] STEP 8B ARTIFACT INVENTORY")

files = []

for p in STEP8B.rglob("*"):
    if not p.is_file():
        continue

    # Do not include the archive currently being generated
    if p.resolve() == ARCHIVE.resolve():
        continue

    # Do not include previous final archive manifests
    if p.resolve() == MANIFEST.resolve():
        continue

    files.append(p)

files = sorted(files)

print(f"Step 8B files found: {len(files)}")

if not files:
    fail("No Step 8B artifacts found.")


# ------------------------------------------------
# 3. HASH ALL ARTIFACTS
# ------------------------------------------------

print("\n[3] HASHING STEP 8B ARTIFACTS")

records = []

for p in files:
    records.append(file_record(p, ROOT))

print(f"Hashed artifacts: {len(records)}")


# ------------------------------------------------
# 4. BUILD MANIFEST
# ------------------------------------------------

print("\n[4] BUILDING FINAL MANIFEST")

manifest_data = {
    "archive_name": ARCHIVE.name,
    "created_utc": datetime.now(timezone.utc).isoformat(),

    "pipeline_step": "8B",
    "pipeline_component": "GeoIP Enrichment + Persistent IP Intelligence",

    "source": {
        "path": str(SOURCE),
        "sha256": source_sha_before,
        "expected_sha256": EXPECTED_SOURCE_SHA,
        "verified": source_sha_before == EXPECTED_SOURCE_SHA,
    },

    "geoip_status": {
        "geoip_database": "MaxMind GeoLite2-City",
        "asn_database": "MaxMind GeoLite2-ASN",
        "persistent_cache": True,
        "external_rdap_resolution": True,
    },

    "artifacts": records,

    "artifact_count": len(records),
}


with open(MANIFEST, "w", encoding="utf-8") as f:
    json.dump(manifest_data, f, indent=2, ensure_ascii=False)

print(f"Manifest: {MANIFEST}")


# ------------------------------------------------
# 5. CREATE FINAL ZIP
# ------------------------------------------------

print("\n[5] CREATING FINAL STEP 8B ARCHIVE")

if ARCHIVE.exists():
    ARCHIVE.unlink()

with zipfile.ZipFile(
    ARCHIVE,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as z:

    # All Step 8B artifacts
    for p in files:
        z.write(
            p,
            arcname=str(p.relative_to(ROOT))
        )

    # Final manifest
    z.write(
        MANIFEST,
        arcname=str(MANIFEST.relative_to(ROOT))
    )

archive_sha = sha256_file(ARCHIVE)

print(f"Archive: {ARCHIVE}")
print(f"Archive size: {ARCHIVE.stat().st_size} bytes")
print(f"Archive SHA-256: {archive_sha}")


# ------------------------------------------------
# 6. ZIP INTEGRITY
# ------------------------------------------------

print("\n[6] ZIP INTEGRITY")

with zipfile.ZipFile(ARCHIVE, "r") as z:
    bad_member = z.testzip()
    members = z.namelist()

if bad_member is not None:
    fail(f"ZIP integrity failure: {bad_member}")

print(f"ZIP members: {len(members)}")
print("ZIP integrity: PASS")


# ------------------------------------------------
# 7. SOURCE PROTECTION AFTER ARCHIVE
# ------------------------------------------------

print("\n[7] FINAL SOURCE PROTECTION")

source_sha_after = sha256_file(SOURCE)

print(f"SHA before: {source_sha_before}")
print(f"SHA after:  {source_sha_after}")

if source_sha_after != source_sha_before:
    fail("Frozen source changed during archive creation.")

print("Source unchanged: PASS")


# ------------------------------------------------
# 8. FINAL HASH RECORD
# ------------------------------------------------

final_record = {
    "pipeline_step": "8B",
    "status": "PASS",

    "source_sha256": source_sha_after,

    "artifact_count": len(records),

    "archive": str(ARCHIVE),
    "archive_size_bytes": ARCHIVE.stat().st_size,
    "archive_sha256": archive_sha,

    "zip_members": len(members),
    "zip_integrity": "PASS",

    "source_protection": "PASS",

    "created_utc": datetime.now(timezone.utc).isoformat(),
}

FINAL_HASH = FINAL_DIR / "STEP_8B_FINAL_HASH.json"

with open(FINAL_HASH, "w", encoding="utf-8") as f:
    json.dump(final_record, f, indent=2, ensure_ascii=False)

print("\n[8] FINAL HASH RECORD")
print(f"Saved: {FINAL_HASH}")


# ------------------------------------------------
# 9. FINAL REPORT
# ------------------------------------------------

print("\n" + "=" * 70)
print("STEP 8B — FINAL ARCHIVE COMPLETE")
print("=" * 70)

print(f"""
Status: PASS

Source SHA-256:
{source_sha_after}

Step 8B artifacts:
{len(records)}

Final archive:
{ARCHIVE}

Archive size:
{ARCHIVE.stat().st_size} bytes

Archive SHA-256:
{archive_sha}

Manifest:
{MANIFEST}

Final hash record:
{FINAL_HASH}

ZIP integrity:
PASS

Source protection:
PASS

Archive integrity:
PASS

NEXT: STEP 8C — ATTACKER IP ANALYTICS
""")

In [ ]:
from google.colab import files
from pathlib import Path

BASE = Path(
    "/content/step8_dashboard_geoip_mitre_session_replay/"
    "8B_geoip_enrichment/final_archive"
)

ARCHIVE = BASE / "STEP_8B_FINAL_ARCHIVE.zip"
MANIFEST = BASE / "STEP_8B_FINAL_ARCHIVE_MANIFEST.json"
HASH_RECORD = BASE / "STEP_8B_FINAL_HASH.json"

required = [ARCHIVE, MANIFEST, HASH_RECORD]

for p in required:
    if not p.exists():
        raise FileNotFoundError(f"Missing required artifact:\n{p}")

print("=" * 70)
print("STEP 8B — DIRECT DOWNLOAD")
print("=" * 70)

print(f"Archive: {ARCHIVE}")
print(f"Manifest: {MANIFEST}")
print(f"Hash record: {HASH_RECORD}")

print("\nStarting downloads...")

files.download(str(ARCHIVE))
files.download(str(MANIFEST))
files.download(str(HASH_RECORD))

print("\n" + "=" * 70)
print("DOWNLOAD REQUESTS SENT")
print("=" * 70)
print("Archive SHA-256:")
print("73ee7f66840bf23efc7552a8cf1eecd4b9fec20419846c4ccd691cab188e6afa")
print("\nSTATUS: PASS")
print("STEP 8B is frozen and archived.")
print("NEXT: STEP 8C — ATTACKER IP ANALYTICS")